In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
train_df = pd.read_csv("data/training_events.csv")
val_df = pd.read_csv("data/validation_events.csv")
test_df = pd.read_csv("data/testing_events.csv")

In [ ]:
train_df.columns

## Reduce embeddings dimensionality with PCA

In [ ]:
embedding_cols = [f"embedding_{i}" for i in range(768)]

X_train_embed = train_df[embedding_cols].values
X_val_embed = val_df[embedding_cols].values
X_test_embed = test_df[embedding_cols].values

In [ ]:
# Apply PCA on transcripts embeddings 

N_COMPONENTS = 50

pca = PCA(n_components=N_COMPONENTS, random_state=42)

X_train_pca = pca.fit_transform(X_train_embed)
X_val_pca = pca.transform(X_val_embed)
X_test_pca = pca.transform(X_test_embed)

print(f"PCA components: {pca.n_components_}")
print(f"Variance preserved: {pca.explained_variance_ratio_.sum():.2%}")

## Prepare training features and target

In [ ]:
market_cols = ["pre_volatility", "pre_momentum", "pre_avg_volume", "earnings_gap"]
X_train_market = train_df[market_cols].values
X_val_market = val_df[market_cols].values
X_test_market = test_df[market_cols].values

In [ ]:
X_train = np.hstack([X_train_pca, X_train_market])
X_val = np.hstack([X_val_pca, X_val_market])
X_test = np.hstack([X_test_pca, X_test_market])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
y_train = train_df["label_3d"].values
y_val = val_df["label_3d"].values
y_test = test_df["label_3d"].values

In [ ]:
print("Final feature shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## Logisitic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

In [ ]:
train_pred = model.predict(X_train_scaled)
val_pred = model.predict(X_val_scaled)
test_pred = model.predict(X_test_scaled)

train_prob = model.predict_proba(X_train_scaled)[:, 1]
val_prob = model.predict_proba(X_val_scaled)[:, 1]
test_prob = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, name):
    print(f"\n{name}")
    print("-" * 30)
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1:        {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_true, y_prob):.4f}")

In [ ]:
evaluate_model(y_train, train_pred, train_prob, "Training")
evaluate_model(y_val, val_pred, val_prob, "Validation")
evaluate_model(y_test, test_pred, test_prob, "Test")

## Experiments w/. different number of principal components

In [ ]:
N_COMPONENTS_LIST = [10, 20, 50, 100, 200]

for n_components in N_COMPONENTS_LIST:

    print(f"\n===== {n_components} PCA components =====")

    # PCA
    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_embed)
    X_val_pca = pca.transform(X_val_embed)
    X_test_pca = pca.transform(X_test_embed)

    print(f"Variance preserved: "f"{pca.explained_variance_ratio_.sum():.2%}")

    # Combine with market features
    X_train = np.hstack([X_train_pca, X_train_market])
    X_val = np.hstack([X_val_pca, X_val_market])
    X_test = np.hstack([X_test_pca, X_test_market])

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Logistic regression
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Predictions
    val_prob = model.predict_proba(X_val_scaled)[:, 1]
    val_pred = model.predict(X_val_scaled)

    print(f"Validation ROC-AUC: {roc_auc_score(y_val, val_prob):.4f}")
    print(f"Validation F1:      {f1_score(y_val, val_pred):.4f}")